In [89]:
import sklearn
from sklearn import svm
import pandas as pd
import numpy as np
import spacy
import nltk
from nltk.util import ngrams

In [90]:
train_path='../data/desegma-it.subTaskA.shared.train.0923-1220.csv'
test_path='../data/desegma-it.subTaskA.shared.test.1117_1835.csv'
test_with_labels_path='../data/desegma-it.subTaskA.with_labels.test.1117_1835.csv'

In [91]:
# 1. Carica il file di training originale
df_full = pd.read_csv(train_path)

# 2. Separa le due classi
# Assumendo che la colonna con le etichette si chiami 'label' 
# (controlla il nome esatto nel tuo file!)
df_class_0 = df_full[df_full['label'] == 0]
df_class_1 = df_full[df_full['label'] == 1]

# 3. Estrai 1000 campioni casuali da ciascuna
train_0 = df_class_0.sample(n=1000, random_state=42)
train_1 = df_class_1.sample(n=1000, random_state=42)

# 4. Uniscili in un unico training set
df_train_final = pd.concat([train_0, train_1])

# 5. Mischia le righe (shuffling) 
# Altrimenti avrai i primi 1000 umani e gli ultimi 1000 IA
df_train_final = df_train_final.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Training set creato: {len(df_train_final)} documenti.")

Training set creato: 2000 documenti.


Attenzione: Evita il "Data Leakage"

Dato che devi estrarre anche 1000 documenti per il Validation Set dallo stesso file di training, non puoi semplicemente ripetere l'operazione. Se lo facessi, rischieresti che alcuni documenti finiscano sia nel training che nel validation, "truccando" i risultati.

Il modo corretto è questo:

    Estrai i 2000 per il Training.

    Rimuovi quei 2000 dal dataset originale.

    Estrai i 1000 per il Validation da quello che resta.

In [92]:
# --- A. ESTRAZIONE VALIDATION (dal residuo del training) ---

# Creiamo un dataframe con quello che RESTA del file originale 
# escludendo gli indici già usati per df_train_final
df_remaining = df_full.drop(df_train_final.index)

# Separiamo per classi nel residuo
df_rem_0 = df_remaining[df_remaining['label'] == 0]
df_rem_1 = df_remaining[df_remaining['label'] == 1]

# Estraiamo 500 e 500 per fare i 1000 di Validation
val_0 = df_rem_0.sample(n=500, random_state=42)
val_1 = df_rem_1.sample(n=500, random_state=42)

df_val_final = pd.concat([val_0, val_1]).sample(frac=1, random_state=42).reset_index(drop=True)


# --- B. ESTRAZIONE TEST (dal file di test con label) ---

df_test_full = pd.read_csv(test_with_labels_path)

# Separiamo le classi nel file di test
df_test_0 = df_test_full[df_test_full['label'] == 0]
df_test_1 = df_test_full[df_test_full['label'] == 1]

# Estraiamo 500 e 500 per fare i 1000 di Test
test_0 = df_test_0.sample(n=500, random_state=42)
test_1 = df_test_1.sample(n=500, random_state=42)

df_test_final = pd.concat([test_0, test_1]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Validation set: {len(df_val_final)} doc (500 class 0, 500 class 1)")
print(f"Test set: {len(df_test_final)} doc (500 class 0, 500 class 1)")

Validation set: 1000 doc (500 class 0, 500 class 1)
Test set: 1000 doc (500 class 0, 500 class 1)


## Data Preparation - Preprocessing

In [ ]:
#!python -m spacy download it_core_news_sm
nlp = spacy.load("it_core_news_sm", disable=["parser", "ner"])
doc = nlp("Apple is looking at buying U.K. startup for $1 billion")
for token in doc:
    print(token.text, token.pos_, token.lemma_)

In [ ]:
print(nlp.pipe_names)

In [ ]:
""" full_words = []
full_pos = []
full_lemma = []
for doc in df_train_final['text'][:5]:
    processed_text=nlp(doc)
    words=[word.pos_ for word in processed_text]
    pos=[word.pos_ for word in processed_text]
    lemma=[word.lemma_ for word in processed_text]
    full_words.
    

df_train_final["words"]=full_words
df_train_final["pos"]=full_pos
df_train_final["lemma"]=full_lemma """

In [ ]:
df_train_final.head()

In [ ]:
df_train_final.iloc[:,0]

In [ ]:
# 1. Carica il modello spacy (assicurati di averlo scaricato: python -m spacy download it_core_news_sm)
nlp = spacy.load("it_core_news_sm")

# Liste temporanee per raccogliere i dati
tokens_list = []
pos_list = []
lemmas_list = []
bigrams_list = []

# 2. Elaborazione con nlp.pipe (più veloce del ciclo for riga per riga)
# Disabilitiamo 'ner' e 'parser' se non servono, per aumentare la velocità
for doc in nlp.pipe(df_train_final['text'], disable=["ner", "parser"]):
    
    # Estraiamo i dati filtrando punteggiatura e stopwords (opzionale ma consigliato)
    # Usiamo .lower() per standardizzare tutto
    t_row = [token.text.lower() for token in doc if not token.is_punct]
    p_row = [token.pos_ for token in doc if not token.is_punct]
    l_row = [token.lemma_.lower() for token in doc if not token.is_punct]
    
    # Creazione dei bigrammi dai lemmi (es: ["il_gatto", "gatto_corre"])
    # Il join con "_" serve a rendere il bigramma un'unica stringa
    bg_row = ["_".join(bg) for bg in ngrams(l_row, 2)]
    
    # Appendiamo i risultati alle liste
    tokens_list.append(t_row)
    pos_list.append(p_row)
    lemmas_list.append(l_row)
    bigrams_list.append(bg_row)

# 3. Popolamento del DataFrame (Assegnazione multipla)
df_train_final['tokens'] = tokens_list
df_train_final['pos'] = pos_list
df_train_final['lemmas'] = lemmas_list
df_train_final['bigrams'] = bigrams_list

# 4. Opzionale: Creiamo una versione "stringa" dei lemmi per il TF-IDF
df_train_final['lemmas_processed'] = df_train_final['lemmas'].apply(lambda x: " ".join(x))

In [93]:
# Carichiamo il modello una sola volta all'inizio dello script
nlp = spacy.load("it_core_news_sm")

def preprocess_dataframe(df, text_column='text'):
    """
    Esegue il preprocessing NLP completo su una colonna di un DataFrame.
    Aggiunge le colonne: tokens, pos, lemmas, bigrams, lemmas_processed.
    """
    # Liste temporanee
    tokens_list = []
    pos_list = []
    lemmas_list = []
    bigrams_list = []

    print(f"Elaborazione in corso per {len(df)} righe...")

    # Batch processing con nlp.pipe
    for doc in nlp.pipe(df[text_column], disable=["ner", "parser"]):
        
        # 1. Estrazione dati base (filtrando punteggiatura)
        t_row = [token.text.lower() for token in doc if not token.is_punct]
        p_row = [token.pos_ for token in doc if not token.is_punct]
        l_row = [token.lemma_.lower() for token in doc if not token.is_punct]
        
        # 2. Creazione Bigrammi dai lemmi
        bg_row = ["_".join(bg) for bg in ngrams(l_row, 2)]
        
        # Append alle liste
        tokens_list.append(t_row)
        pos_list.append(p_row)
        lemmas_list.append(l_row)
        bigrams_list.append(bg_row)

    # 3. Assegnazione multipla al DataFrame originale
    # Usiamo .assign o l'assegnazione diretta. Per chiarezza:
    df['tokens'] = tokens_list
    df['pos'] = pos_list
    df['lemmas'] = lemmas_list
    df['bigrams'] = bigrams_list
    
    # 4. Creazione versione stringa per TF-IDF
    df['lemmas_processed'] = df['lemmas'].apply(lambda x: " ".join(x))
    
    print("Completato!")
    return df

In [94]:
# --- UTILIZZO ---

df_train_final = preprocess_dataframe(df_train_final)
df_test_final = preprocess_dataframe(df_test_final)
df_val_final = preprocess_dataframe(df_val_final)

Elaborazione in corso per 2000 righe...
Completato!
Elaborazione in corso per 1000 righe...
Completato!
Elaborazione in corso per 1000 righe...
Completato!


In [95]:
# Visualizza il risultato
df_train_final.head()

,text,label,tokens,pos,lemmas,bigrams,lemmas_processed
0,"Eppure, nel mezzo di queste scorse commerciali...",1,"[eppure, nel, mezzo, di, queste, scorse, comme...","[CCONJ, ADP, NOUN, ADP, DET, NOUN, ADJ, DET, A...","[eppure, in il, mezzo, di, questo, scorsa, com...","[eppure_in il, in il_mezzo, mezzo_di, di_quest...",eppure in il mezzo di questo scorsa commercial...
1,"""È una cosa che sto valutando di fare da fine ...",0,"[è, una, cosa, che, sto, valutando, di, fare, ...","[AUX, DET, NOUN, PRON, AUX, VERB, ADP, VERB, A...","[essere, uno, cosa, che, stare, valutare, di, ...","[essere_uno, uno_cosa, cosa_che, che_stare, st...",essere uno cosa che stare valutare di fare da ...
2,Il caso Pirelli si presenta non come un sempli...,1,"[il, caso, pirelli, si, presenta, non, come, u...","[DET, NOUN, PROPN, PRON, VERB, ADV, ADP, DET, ...","[il, caso, pirelli, si, presentare, non, come,...","[il_caso, caso_pirelli, pirelli_si, si_present...",il caso pirelli si presentare non come uno sem...
3,Poi la direzione distrettuale antimafia di Lec...,0,"[poi, la, direzione, distrettuale, antimafia, ...","[ADV, DET, NOUN, ADJ, ADJ, ADP, PROPN, ADP, DE...","[poi, il, direzione, distrettuale, antimafia, ...","[poi_il, il_direzione, direzione_distrettuale,...",poi il direzione distrettuale antimafia di lec...
4,Il cambio di programma si è configurato in una...,1,"[il, cambio, di, programma, si, è, configurato...","[DET, NOUN, ADP, NOUN, PRON, AUX, VERB, ADP, D...","[il, cambio, di, programma, si, essere, config...","[il_cambio, cambio_di, di_programma, programma...",il cambio di programma si essere configurare i...


In [96]:
df_test_final.head()

,text,label,tokens,pos,lemmas,bigrams,lemmas_processed
0,"Quaranta, si dice, ma forse sono molti di più,...",1,"[quaranta, si, dice, ma, forse, sono, molti, d...","[NUM, PRON, VERB, CCONJ, ADV, AUX, PRON, ADP, ...","[quaranta, si, dire, ma, forse, essere, molto,...","[quaranta_si, si_dire, dire_ma, ma_forse, fors...",quaranta si dire ma forse essere molto di più ...
1,Una notizia che arriva proprio mentre la Corte...,1,"[una, notizia, che, arriva, proprio, mentre, l...","[DET, NOUN, PRON, VERB, ADV, SCONJ, DET, NOUN,...","[uno, notizia, che, arrivare, proprio, mentre,...","[uno_notizia, notizia_che, che_arrivare, arriv...",uno notizia che arrivare proprio mentre il cor...
2,Uno studio recente della Harvard School of Pub...,1,"[uno, studio, recente, della, harvard, school,...","[DET, NOUN, ADJ, ADP, PROPN, PROPN, ADP, PROPN...","[uno, studio, recente, di il, harvard, school,...","[uno_studio, studio_recente, recente_di il, di...",uno studio recente di il harvard school of pub...
3,""",\n Come ha fatto sapere stamane il president...",1,"[\n , come, ha, fatto, sapere, stamane, il, pr...","[SPACE, SCONJ, AUX, VERB, VERB, ADV, DET, NOUN...","[\n , come, avere, fare, sapere, stamane, il, ...","[\n _come, come_avere, avere_fare, fare_sapere...",\n come avere fare sapere stamane il presiden...
4,Questa volta a far scattare il duello tutto in...,0,"[questa, volta, a, far, scattare, il, duello, ...","[DET, NOUN, ADP, VERB, VERB, DET, NOUN, PRON, ...","[questo, volta, a, fare, scattare, il, duello,...","[questo_volta, volta_a, a_fare, fare_scattare,...",questo volta a fare scattare il duello tutto i...
